## Step 1: OCR ##
The first step is to download paddleOCR and test it on some of the dataset. PaddleOCR is ran with *PaddlePaddle* as their deep learning model so it's important to download both libraries together

Source: https://github.com/PaddlePaddle/PaddleOCR

For CPU-only PaddlePaddle:
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

In [1]:
from paddleocr import PaddleOCR

c:\Users\alexr\anaconda3\envs\MLP\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\alexr\anaconda3\envs\MLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [4]:
ocr = PaddleOCR(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)
result = ocr.predict(input="raw_pdf/electric bills-part-1.pdf")

for page in result:
  page.save_to_img("pages")
  page.save_to_json("output")


Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alexr\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alexr\.paddlex\official_models\PP-OCRv5_server_rec`.


## Step 2: Data Preprocessing ##
We've now saved our page into their folders pages and outputs, saving the images and json files respectively. <br><br>Utility companies send out bills with multiple pages of information. It would be a waste to train our model on all those pages where classification isn't needed. Similar to **Term Frequency-Inverse Document Frequency (TF-IDF)**, each page will be read to count through how many lines, words, and digits to determine which page is likely to have the itemized bill. We will move this file into its own folder to train and test on.

In [27]:
import os, json, re, shutil, csv
from pathlib import Path

OCR_DIR = "output"
PAGES_DIR = "pages"
TRAIN_IMG = "training_data/images"
TRAIN_ANN = "training_data/annotation_raw"
TRAIN_WRD = "training_data/annotation_words"
LOG_DIR = "logs"

KEYWORDS = {"energy": [r"kwh", r"kilowatt.?hour"],
            "demand": [r"kw\b", r"peak\s+demand"],
            "cost": [r"charge"]}

num_re = re.compile(r"^-?\d+(\.\d+)?$")

In [ ]:
# Function to look for digits
def is_numeric_token(token):
    # strip common non-numeric chars (currency, commas, percent, parentheses)
    cleaned = re.sub(r"[^\d\.\-]", "", token)
    return bool(cleaned) and bool(num_re.match(cleaned))

# Function to extract text from each line
def extract_text_lines(data):
    if isinstance(data, dict):
        return data.get("rec_texts") or data.get("texts") or []
    if isinstance(data, list):
        # maybe list of [box, text] or [[x1,y1,...], "text"]
        texts = []
        for it in data:
            if isinstance(it, list) and len(it) >= 2 and isinstance(it[-1], str):
                texts.append(it[-1])
        return texts
    return []

# Function to compare tokens to the ones we want
def is_keyword_token(tokens):
    # make all tokens lower case to find correct match
    token_lower = tokens.lower()
    # Loop through dictionary
    for _, value in KEYWORDS.items():
        for keyword in value:
            # Compare values in dictionary
            if re.search(keyword, token_lower):
                return True
    return False
    
def score_page(lines):
    # Store variables
    keyword_token = 0
    total_words = 0
    numeric_words = 0

    for line in lines:
        # simple tokenization on whitespace; adjust if you need finer splitting
        tokens = re.findall(r"\S+", line)
        total_words += len(tokens)
        # Count number of digits in tokens
        for t in tokens:
            if is_numeric_token(t):
                numeric_words += 1
            if is_keyword_token(t):
                keyword_token += 1
    keyword_density = keyword_token / total_words
    numeric_density = numeric_words / total_words

    return keyword_density, numeric_density

# Save score values per page
scores = []
# Loop through each page in the output
for fname in sorted(os.listdir(OCR_DIR)):
    path = os.path.join(OCR_DIR, fname)
    # Open file to read
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    lines = extract_text_lines(data)
    keyword_density, numeric_density = score_page(lines)
    print(f"{fname}: lines={len(lines)}, keyword_density={keyword_density:.4f}, numeric_density={numeric_density:.4f}")
    scores.append({
        "fname": fname,
        "keyword_density": keyword_density,
        "numeric_density": numeric_density,
    })

best_page = max(scores, key=lambda x: x["keyword_density"])


electric bills-part-1_0_res.json: lines=114, keyword_density=0.0107, numeric_density=0.1744
electric bills-part-1_1_res.json: lines=120, keyword_density=0.0560, numeric_density=0.0917
electric bills-part-1_2_res.json: lines=18, keyword_density=0.0435, numeric_density=0.0326
electric bills-part-1_1_res.json


After determining which page is the one holding our bill. We will move these into the training dataset to train our model on.

In [32]:
# Maintain base form of file to move
base = best_page['fname'].replace("_res.json","")
img_src = os.path.join(PAGES_DIR, base+"_ocr_res_img.png")
ann_src = os.path.join(OCR_DIR, best_page['fname'])

if best_page:
  # Move JSON to annotations
  ann_dst = os.path.join(TRAIN_ANN, best_page['fname'])
  shutil.copy2(ann_src, ann_dst)

  # Move image to training images
  if os.path.exists(img_src):
    img_dst = os.path.join(TRAIN_IMG, base + ".png")
    shutil.copy2(img_src, img_dst)


Now it's crucial we build our labels for LayoutLM accordingly

In [ ]:
split_re = re.compile(r"\S+")

def build_page_entry(page_id, all_tokens, all_bboxes, all_labels):
  if len(all_tokens) > 510:
    print(f"WARN {page_id}: {len(all_tokens)} tokens exceeds limit, will be truncated")

  return {
    "id": page_id,
    "tokens": all_tokens,
    "bboxes": all_bboxes,
    "labels": all_labels,
    "token_count": len(all_tokens)   # useful to log
  }

def label_line(tokens):
    labels = []
    numeric_index = 0
    line_has_keyword = any(is_keyword_token(t) for t in tokens)
    
def poly_to_bbox(poly):
    xs = [p[0] for p in poly]
    ys = [p[1] for p in poly]
    return [min(xs), min(ys), max(xs), max(ys)]

def split_bbox_horizontally(bbox, words):
    x0, y0, x1, y1 = bbox
    total_chars = sum(len(w) for w in words) if words else 1
    bboxes = []
    cur_x = x0
    width = x1 - x0
    for w in words:
        # allocate proportional width
        frac = len(w) / total_chars
        w_w = max(1, round(frac * width))
        x_start = cur_x
        x_end = x_start + w_w
        if x_end > x1:
            x_end = x1
        bboxes.append([int(x_start), int(y0), int(x_end), int(y1)])
        cur_x = x_end
    # fix last bbox to end exactly at x1
    if bboxes:
        bboxes[-1][2] = int(x1)
    return bboxes

def convert_file(input_path, out_fh):
    obj = json.loads(input_path.read_text())
    rec_texts = obj.get("rec_texts", [])
    rec_polys = obj.get("rec_polys", [])
    rec_scores = obj.get("rec_scores", [])
    page_index = obj.get("page_index", 0)
    source = obj.get("input_path") or input_path.name

    for i, text in enumerate(rec_texts):
        poly = rec_polys[i] if i < len(rec_polys) else None
        score = rec_scores[i] if i < len(rec_scores) else None
        if text is None:
            continue
        words = split_re.findall(text)
        if not words:
            continue
        if poly:
            bbox = poly_to_bbox(poly)
            word_bboxes = split_bbox_horizontally(bbox, words)
        else:
            word_bboxes = [[0,0,0,0] for _ in words]

        out = {
            "id": f"{Path(input_path).stem}_p{page_index}_l{i}",
            "source": source,
            "line_index": i,
            "raw_text": text,
            "tokens": words,
            "bboxes": word_bboxes,
            "line_score": score,
        }
        out_fh.write(json.dumps(out, ensure_ascii=False) + "\n")

input_file = Path(TRAIN_ANN) / best_page['fname']
out_path = Path(TRAIN_WRD) /  (base + ".jsonl")

# files = sorted(input_file.glob("*_res.json"))
with out_path.open("w", encoding="utf-8") as fh:
    convert_file(input_file, fh)

print(f"Wrote HF jsonl to: {out_path}")

Wrote HF jsonl to: training_data\annotation_words\electric bills-part-1_1.jsonl


## Step 3: LayoutLMv3 ##
Now comes our model we are going to use for transfer learning. LayoutLM extends the traditional trasnformer model by integrating layout information into the input embeddings designed for document AI tasks that can model texts, images and layout together. It will be the backbone of our project. Now that we have filtered our data into each respective folder, let's pass it into our model to fine-tune it.

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)